# مسار ومختبر تحليل البيانات المتكامل (Data Analytics Mastery 2026)
**إشراف وتصميم: د. محمد عبدالمنعم لبد (Dr. Mohammed Lubbad · Sky Develop Venture Studio)**

مرحباً بك في الدفتر التطبيقي الموحد للكورس. صممتُ لك هذا المختبر على غرار قضايا التحقيق الواقعية (`Detective Cases`) لتمارس العمل الفعلي لمحلل البيانات من صياغة سؤال العمل وتطبيق بوابات التحقق السبع، مروراً ببناء استعلامات `SQL` المتقدمة وحل معضلة تضخم الإيرادات عند الربط (`Join Fan-out`)، ووصولاً إلى المعالجة المتجهة في `Pandas` والسرد القصصي التنفيذي وفق منهجية `Cole Nussbaumer Knaflic`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sky-develop/data-venture-studio/blob/main/courses/data-analytics-mastery-2026/labs/Data_Analytics_Masterclass_Lab_01_to_06.ipynb)

## الوحدة 1: هندسة سؤال العمل وبوابات التحقق السبع وحوكمة البيانات (DIKW & 7 Validation Gates)
**القضية الأولى:** يدّعي تقرير المبيعات الأولي لمتجر «سحاب الخليج» تحقيق إيرادات قدرها **2,163,236.09 ريال**، بينما يشتكي المدير المالي من فجوة نقدية قدرها **581,064.27 ريال**. قبل كتابة أي معادلة إحصائية، نطبق بوابات التحقق السبع (من الكتاب المرجعي `#001`) مع تعمية بيانات العملاء الحساسة (`PII Masking` وفق ضوابط `SDAIA / NDMO`).

In [1]:
import os, sqlite3, hashlib, json, urllib.request
import pandas as pd
import numpy as np

# تحميل قاعدة البيانات تلقائياً عند التشغيل داخل Google Colab أو محلياً
db_local = '../datasets/gcc_analytics_warehouse.db'
if not os.path.exists(db_local):
    db_local = 'gcc_analytics_warehouse.db'
    if not os.path.exists(db_local):
        urllib.request.urlretrieve('https://training.sky-develop.net/academy/datasets/sahab-gcc-commerce/gcc_analytics_warehouse.db', db_local)

conn = sqlite3.connect(db_local)
df_orders_raw = pd.read_sql_query('SELECT * FROM raw_orders', conn)
df_cust_raw = pd.read_sql_query('SELECT * FROM raw_customers', conn)

# 1. تعمية أرقام الهواتف والبريد الإلكتروني (SDAIA PII Compliance)
df_cust_raw['phone_hash'] = df_cust_raw['phone_raw'].apply(lambda x: hashlib.sha256(('sky_salt_' + str(x)).encode()).hexdigest()[:12])
df_cust_safe = df_cust_raw.drop(columns=['phone_raw', 'email_raw'])

# 2. فحص بوابات التحقق السبع (7 Data Validation Gates)
print('إجمالي الصفوف الخام:', len(df_orders_raw))
print('بوابة التفرّد (Duplicate Order IDs):', df_orders_raw.duplicated(subset=['order_id']).sum())
print('بوابة النطاق الزمني (Future Dates 2099):', (df_orders_raw['order_date_raw'] == '2099-01-01').sum())
print('بوابة النطاق الكمي (Negative/Zero Qty):', (df_orders_raw['quantity'] <= 0).sum())
print('الطلبات المستردة المسجلة كموجبة:', (df_orders_raw['status'] == 'REFUNDED').sum())

## الوحدة 2: SQL التحليلية والنمذجة البعدية وحل فخ تكرار الربط (Star Schema & Join Fan-out)
**القاعدة الذهبية (Ralph Kimball `#013` + Felienne Hermans):** لا تربط جدولين قبل إعلان الحبيبية (`Declare the Grain`). جدول `raw_orders` حبيبيته `صف لكل طلب`، بينما جدول `raw_payment_attempts` حبيبيته `صف لكل محاولة دفع`. الربط المباشر قبل التجميع يضاعف الإيرادات الوهمية بنسبة **36.7%**!

In [2]:
# استعلام SQL مرحلي (CTEs) يفصل التجميع قبل الربط ويوحد العملات إلى الريال السعودي (SAR)
sql_star_audit = '''
WITH dedup_orders AS (
    SELECT DISTINCT order_id, customer_id, order_date_raw, status, quantity, unit_price_raw, currency, order_amount_raw
    FROM raw_orders
    WHERE quantity > 0
      AND status = 'COMPLETED'
      AND order_date_raw != '2099-01-01'
),
captured_payments AS (
    SELECT order_id,
           COUNT(*) AS total_gateway_attempts,
           SUM(CASE WHEN attempt_status = 'CAPTURED' THEN 1 ELSE 0 END) AS captured_count
    FROM raw_payment_attempts
    GROUP BY order_id
    HAVING captured_count >= 1
)
SELECT o.order_id, o.customer_id, o.currency, o.order_amount_raw,
       p.total_gateway_attempts,
       ROUND(o.order_amount_raw * CASE
           WHEN o.currency = 'SAR' THEN 1.0
           WHEN o.currency = 'AED' THEN 1.02
           WHEN o.currency = 'OMR' THEN 9.74
           ELSE 1.0 END, 2) AS net_revenue_sar
FROM dedup_orders o
JOIN captured_payments p ON o.order_id = p.order_id
'''
df_clean = pd.read_sql_query(sql_star_audit, conn)
print('عدد الطلبات الصحيحة المعتمدة:', len(df_clean))
print('صافي الإيرادات الحقيقية (SAR):', round(df_clean['net_revenue_sar'].sum(), 2))

## الوحدة 3: بايثون المتجهة وتوحيد البيانات العربية (Vectorized Pandas & Tidy Data)
**القاعدة الذهبية (Wes McKinney `#002` & Hadley Wickham `#004`):** يمنع استخدام حلقات `for` أو `.iterrows()` على الجداول. نستخدم العمليات المتجهة (`Vectorized String & Map Operations`) لتوحيد أسماء المدن العربية والإنجليزية وتواريخ الطلبات.

In [3]:
city_canonical_map = {
    'الرياض': 'الرياض (Riyadh)', 'Riyadh': 'الرياض (Riyadh)', ' الرياض ': 'الرياض (Riyadh)', 'أر رياض': 'الرياض (Riyadh)',
    'جدة': 'جدة (Jeddah)', 'Jeddah': 'جدة (Jeddah)',
    'الدمام': 'الدمام (Dammam)', 'Dammam': 'الدمام (Dammam)',
    'دبي': 'دبي (Dubai)', 'Dubai': 'دبي (Dubai)',
    'مسقط': 'مسقط (Muscat)', 'Muscat': 'مسقط (Muscat)'
}
df_cust_safe['city_clean'] = df_cust_safe['city_raw'].map(city_canonical_map).fillna('غير محدد')
df_cust_safe['segment_clean'] = df_cust_safe['segment'].fillna('Unclassified_Tier')
df_mart = df_clean.merge(df_cust_safe, on='customer_id', how='left')
print(df_mart.groupby('city_clean')['net_revenue_sar'].agg(['count', 'sum', 'median']).sort_values(by='sum', ascending=False))

## الوحدة 4: الإحصاء التطبيقي وكشف التواء المتوسط الحسابي (Mean vs. Median & Whale Outliers)
**القاعدة الذهبية:** في توزيعات المبيعات الملتوية يميناً (`Right-Skewed Distributions`)، المتوسط الحسابي (`1,091.15 ريال`) يتأثر بـ 12 صفقة مؤسسية ضخمة (`Whale Orders`)، بينما الوسيط الحسابي (`430.12 ريال`) يعكس سلوك العميل المعتاد بدقة.

In [4]:
mean_sar = df_mart['net_revenue_sar'].mean()
median_sar = df_mart['net_revenue_sar'].median()
iqr_sar = df_mart['net_revenue_sar'].quantile(0.75) - df_mart['net_revenue_sar'].quantile(0.25)
print(f'المتوسط الحسابي المشوّه بالصفقات الضخمة: {mean_sar:,.2f} SAR')
print(f'الوسيط الحسابي المتين (Median): {median_sar:,.2f} SAR | المدى الربيعي (IQR): {iqr_sar:,.2f} SAR')

## الوحدة 5 و 6: السرد القصصي التنفيذي (Knaflic SWD) وتوصية القرار
**العنوان التنفيذي (Action Title):** «تضخم الإيرادات المبلّغ عنها بنسبة 36.7% (581 ألف ريال) ناتج عن ربط محاولات الدفع المتكررة قبل التجميع، وليس نمواً في المبيعات».